# Raport z Projektu: Prevent Neural Networks from Overfitting

---

## 1. Wstęp

Zjawisko **overfittingu** (przeuczenia) jest jednym z głównych problemów przy trenowaniu głębokich sieci neuronowych. Występuje ono, gdy model zbytnio adaptuje się do danych treningowych, ucząc się "na pamięć" szumu i nieistotnych wzorców, co drastycznie obniża jego zdolność do generalizacji na nowych danych.

W 2014 roku Srivastava et al. w publikacji *Dropout: A Simple Way to Prevent Neural Networks from Overfitting* wprowadzili innowacyjną technikę regularyzacji. **Dropout** polega na losowym "wyłączaniu" (zerowaniu) aktywacji pewnej części neuronów podczas treningu (z prawdopodobieństwem $p$). Uniemożliwia to neuronom ślepą ko-adaptację i zmusza każdy z nich do niezależnego wyodrębniania użytecznych cech.

## 2. Metodologia (Reprodukcja Baseline)

W ramach pierwszej części projektu odtworzono środowisko testowe z klasycznej publikacji dla zbioru danych **MNIST** (odręcznie pisane cyfry).

Zaimplementowano Wielowarstwowy Perceptron (MLP) o następującej architekturze:
- **Warstwa wejściowa:** 784 neurony (28x28 pikseli).
- **Warstwy ukryte:** Dwie warstwy po 1024 neurony z funkcją aktywacji ReLU.
- Zastosowano Dropout po każdej warstwie ukrytej.
- **Warstwa wyjściowa:** 10 neuronów (klasyfikacja 10 klas).

Przeprowadzono trening dla trzech wariantów:
- $p=0.0$ (Brak Dropoutu - Baseline)
- $p=0.2$ (Lekki Dropout)
- $p=0.5$ (Standardowy Dropout opisany w publikacji).

## 3. Eksperyment Własny ("Wow Angle")

Głównym pytaniem badawczym projektu jest: 
**"Czy w 2026 roku, dysponując nowoczesnymi technikami regularyzacji, używanie klasycznego Dropoutu jest wciąż absolutnie niezbędne dla podstawowego MLP?"**

Aby to sprawdzić, usunięto całkowicie Dropout ($p=0$) i zastąpiono go kombinacją dwóch technik:
1. **L2 Regularization (Weight Decay):** Zastosowano w optymalizatorze `AdamW`. Mechanizm ten karze sieć za posiadanie zbyt dużych i zbytnio dostosowanych wag. Zbadano parametry z zakresu $[1e-5, 1e-3, 1e-2]$, gdzie optymalną wartością po eksperymentach okazało się `0.01`.
2. **Early Stopping:** Mechanizm dynamicznego przerywania treningu. Kiedy błąd na zbiorze walidacyjnym (Validation Loss) przestaje maleć przez określoną liczbę epok (cierpliwość/patience = 7), trening jest przerywany, co skutecznie zapobiega wkroczeniu w fazę przeuczenia.

## 4. Wyniki i Dyskusja

### Reprodukcja Baseline:

<img src="./results/baseline_dropout.png" alt="Baseline Dropout" width="700"/>

Analiza pierwszego wykresu pozwala na sformułowanie następujących obserwacji:
- Model bez Dropoutu ($p=0.0$) początkowo uczy się niezwykle szybko, jednak po pewnym czasie błąd na zbiorze testowym zaczyna stopniowo rosnąć - jest to podręcznikowy przykład przeuczania.
- Zastosowanie Dropoutu znacząco stabilizuje ten proces. Wariant klasyczny $p=0.5$, pomimo wolniejszego uczenia w początkowych epokach, pozwala na osiągnięcie długoterminowo płaskiej i optymalnej linii błędu, zachowując wysoką zdolność do generalizacji.

---

### Eksperyment Wow Angle (Nowoczesna Regularyzacja):

<img src="./results/wow_angle_comparison.png" alt="Wow Angle Comparison" width="700"/>

Wykres ten bezpośrednio konfrontuje metody klasyczne z nowoczesnym podejściem:
- Krzywa reprezentująca `AdamW WD=0.01 + ES` (bez użycia Dropoutu) charakteryzuje się bardzo rewelacyjnym i gładkim zejściem błędu testowego.
- Algorytm korzystający z optymalnego Weight Decay był w stanie w pewnym momencie wygenerować błąd niższy niż w przypadku standardowego Dropoutu ($p=0.5$). Mechanizm Early Stopping zadbał o to, by przerwać trening w idealnym momencie, zachowując najlepsze wagi modelu.

## 5. Wnioski

Eksperyment powiódł się i doprowadził do jasnych konkluzji.

Choć **Dropout** był techniką wysoce rewolucyjną i w 2014 roku całkowicie zmienił podejście do głębokich sieci neuronowych, w **2026 roku nie jest on już jedynym ostatecznym rozwiązaniem** dla podstawowych architektur typu MLP.

Odpowiednie i staranne dostrojenie nowoczesnego **Weight Decay (w optymalizatorze AdamW)** w rzetelnej asyście mechanizmu **Early Stopping** pozwala wyeliminować problem overfittingu z niemal identyczną, a czasami przewyższającą skutecznością. Dowodzi to tezy, że współczesny inżynier uczenia maszynowego dysponuje na tyle bogatym arsenałem narzędzi stabilizujących trening, że w wielu przypadkach poleganie na samym Dropoucie przestaje być rynkowym przymusem.